# An AI SAT Tutor and an LLM-as-Judge Evaluation Benchmark

Two linked pieces of applied-AI work:

1. **A Socratic tutor bot** — an LLM (`gpt-5-nano`) prompted to help low-income high-school
   students prepare for the SAT *without ever handing over the answer*, using hints and
   checkpoint questions instead.
2. **An evaluation benchmark** — an **LLM-as-judge** harness that scores the tutor's responses
   against an explicit rubric, so "is this tutor actually good?" becomes measurable rather than
   a matter of vibes.

The design choice at the center is the **alignment trade-off**: a prompt strict enough to never
leak answers can become useless, so the tutor is tuned to stay helpful while resisting
answer-extraction — and the benchmark is built to *detect* when it fails.

> Course project — Harvard Kennedy School, DPI-681 (*Public Problem Solving with Generative AI*).
> Requires `OPENAI_API_KEY`.

## Setup

In [ ]:
# Install dependencies
!pip install openai tqdm pydantic pandas --quiet

import os
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
print("Libraries imported and API client created.")

## The tutor: system prompt

The prompt encodes the pedagogy — patient, not one-size-fits-all, never gives the answer
directly, uses hints + checkpoint questions, and has an explicit escalation path when a student
keeps pushing for the answer.

In [ ]:
# Paste your system prompt here (replace the text below)
SYSTEM_PROMPT = """

You are an AI tutorbot designed to help high school students (eleventh and twelfth graders) in low-income areas prepare for the SAT. You will be split into two modes, SAT Math and SAT Reading/Writing, with one shared core tutor policy. Your primary goal is to help students improve their skills and confidence for standardized tests, and build their problem-solving skills that will also be able to assist them in the classroom.

You should behave like a patient and encouraging tutor that is aimed at meeting the needs of a wide variety of students, rather than following a “one-size-fits-all” approach. You should avoid being overly technical in order not to confuse the students further. You should also make sure not to over-explain. You should never provide the answer directly, no matter how many times students may prompt you for it. You can give hints, but you should also include checkpoint questions in order to keep the students on track. You should periodically generate quizzes based on students’ performances and monitor for their weaknesses, with more emphasis on those areas. You should also include an option for support in other languages if needed.

If a student continues to ask for the answer repeatedly, you should:
Ask the student what they think they should do next
Break the problem down into smaller sections
Provide 1-2 optional hints if still needed (the hint should never give away the answer)
End with a small question at the end to verify understanding

"""

## Conversation handling (multi-turn memory)

In [ ]:
def chat_with_tutor(user_message, conversation_history):
    """
    Send a user message to the tutor bot and get a response.

    Parameters:
    - user_message (str): The message from the student
    - conversation_history (list): List of message dictionaries containing the conversation so far

    Returns:
    - str: The tutor bot's response
    """
    # Add the user message to conversation_history
    # (Hint: use conversation_history.append() with a dictionary with
    # "role": "user" and "content": user_message)

    conversation_history.append({"role":"user",
                                 "content": user_message})

    # Make the API call with the full conversation history
    response = client.chat.completions.create(
        model="gpt-5-nano",  # Specify the model (e.g., "gpt-5-mini")
        messages=conversation_history  # Specify the conversation_history
    )

    # Extract the assistant's response
    response_text = response.choices[0].message.content

    # Add the assistant's response to conversation_history
    # Hint: This is the same as the user message, but with the role "assistant" and a different message

    conversation_history.append({"role":"assistant",
                                 "content": response_text})

    # Return the response text
    return response_text

In [ ]:
# Test your function with a simple conversation
test_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

# First message
response1 = chat_with_tutor("What is 2+2?", test_history)
print("First response received:", len(response1) > 0)

# Second message (should have context from first)
response2 = chat_with_tutor("What about 3+3?", test_history)
print("Second response received:", len(response2) > 0)

# Check that conversation history is being maintained
print("Conversation has", len(test_history), "messages (should be 5: system + 2 user + 2 assistant)")

if len(test_history) == 5:
    print("✅ Success! Your conversation history is being maintained correctly.")
else:
    print("❌ Error: Conversation history not maintained correctly.")

## The benchmark

To evaluate the tutor, I define **test examples**, a **rubric**, a function to **generate** tutor
responses, and an **LLM judge** that scores each response against the rubric (with structured
output so scores are always parseable). Then I run the whole benchmark.

In [ ]:
# Create your list of test examples here
test_examples = [
    # Add your test examples here
    # Each example should be a dictionary with at least a "question" key
    {"question": "Solve for x: 2(x - 3) + 4 = 3(2x + 1)", "topic": "SAT Math"},
    {"question": "If 4x - 2y = 10 and 2x + 5y = 1, what is the value of y?", "topic": "SAT Math"},
    {"question": "Simplify the expression: 3(2x - 4) - 2(3x + 1)", "topic": "SAT Math"},
    {"question": "What does the word 'ambiguous' most nearly mean in this sentence: 'His answer was ambiguous and unclear'?", "topic": "SAT Reading"},
    {"question": "Solve: 2(x - 3) = 10", "topic": "SAT Math"}
    # Add more examples here (aim for 5-10 total)
]

**Evaluation rubric** — four criteria, since a tutor can be accurate yet still fail by giving away the answer:

In [ ]:
# Create your evaluation rubric here
evaluation_rubric = """
1. Accuracy (1-5): Is the SAT math and reading/writing information correct and up-to-date?
2. Clarity (1-5): Is the explanation clear and easy for 11th-12th graders to understand? Avoid overly technical, detailed language.
3. Quality (1-5): Does the tutorbot guide the student through the material instead of just providing the answers? Are the hints helpful without giving too much away?
4. Support (1-5): Is the tutorbot motivating and patient? Does it encourage the student to solve the problem?

Provide a single overall score from 1-5.
"""

# IMPORTANT: Set the maximum score for your rubric scale.

rubric_max_score = 5  # Change this to match your rubric's maximum score

In [ ]:
def generate_tutor_response(test_example):
    """
    Generate a response from your tutor bot for a given test example.

    Parameters:
    - test_example (dict): A test example dictionary with at least a "question" key

    Returns:
    - str: The tutor bot's response
    """
    # Extract the question from test_example

    question = test_example["question"]

    # Create a fresh conversation history with your system prompt
    conversation_history = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]


    # Call chat_with_tutor() to get a response with the question and conversation history
    response = chat_with_tutor(question, conversation_history)

    # Return the response
    return response

In [ ]:
# Define the response structure using Pydantic
# This ensures the judge model always returns a single score number
try:
    from pydantic import BaseModel, ConfigDict, Field
except ImportError:
    print("⚠️  Pydantic not found. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pydantic"])
    from pydantic import BaseModel

class JudgeResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")  # <-- key line
    score: float = Field(..., ge=0)            # optional constraints

# This model will be used automatically - you don't need to worry about it!
print("✅ Response structure defined (this ensures reliable score extraction)")

In [ ]:
def evaluate_response(test_example, tutor_response):
    """
    Use a judge model to evaluate a tutor bot response.

    Parameters:
    - test_example (dict): The original test example
    - tutor_response (str): The tutor bot's response to evaluate

    Returns:
    - float: The judge's score scaled to 0-100
      (automatically scales from the rubric's scale defined by rubric_max_score)

    Note: This function uses Pydantic models and structured outputs to ensure the judge
    always returns a valid score. The student's prompt is automatically enhanced with
    response format instructions, and the response is parsed using a Pydantic model.
    The score is automatically scaled from 0-rubric_max_score to 0-100 for consistent comparison.
    """
    # Extract the question from test_example
    question = test_example['question']

    # Create your evaluation prompt here
    # Write your prompt to evaluate the tutor bot's response
    # Include: the original question, the tutor response, and your rubric
    YOUR_PROMPT = f"""
    Original Question:
    {question}

    Tutor Bot Response:
    {tutor_response}

    Evaluation Rubric:
    {evaluation_rubric}

    Please evaluate the tutor bot's response according to the rubric above.
    Give one overall score from 1 to {rubric_max_score}.

    """


    # The code below automatically handles response formatting and parsing
    # You don't need to modify anything below this line

    judge_system_prompt = "You are an expert evaluator of educational content. You evaluate tutor bot responses according to provided rubrics."

    # Create messages list for the judge
    judge_messages = [
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": YOUR_PROMPT}
    ]


    # Check that rubric_max_score is defined
    if 'rubric_max_score' not in globals():
        raise ValueError("rubric_max_score is not defined. Please set it in Step 2 to match your rubric's maximum score.")

    # Automatically append response format instructions to the user's prompt
    # This ensures the model returns the correct structure without students having to worry about it
    import json

    # Append instructions about the response format to the prompt
    formatted_prompt = YOUR_PROMPT + f"\n\nIMPORTANT: You must respond with a JSON object containing a single 'score' field with a number from 0 to {rubric_max_score}. Example: {{\"score\": 4}}"

    # Update messages with the formatted prompt
    judge_messages_formatted = [
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": formatted_prompt}
    ]

    # Use structured output with Pydantic model to enforce the response shape
    # This ensures we ALWAYS get a valid score, regardless of how the student wrote their prompt
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=judge_messages_formatted,
        response_format={"type": "json_schema", "json_schema": {
            "name": "judge_response",
            "strict": True,
            "schema": JudgeResponse.model_json_schema(),
            "description": "The judge's evaluation score"
        }}
    )

    # Parse the response using Pydantic (this validates the structure automatically)
    evaluation_text = response.choices[0].message.content
    evaluation_json = json.loads(evaluation_text)
    judge_response = JudgeResponse(**evaluation_json)
    raw_score = judge_response.score

    # Validate that the score is within a reasonable range for the rubric
    if raw_score < 0:
        print(f"⚠️  Warning: Judge returned negative score ({raw_score}). Clamping to 0.")
        raw_score = 0
    if raw_score > rubric_max_score * 2:  # Allow some flexibility, but not too much
        print(f"⚠️  Warning: Judge returned score ({raw_score}) much higher than rubric_max_score ({rubric_max_score}).")
        print(f"   This might indicate the judge misunderstood the rubric scale.")

    # Scale the score from 0-rubric_max_score to 0-100
    # Formula: (raw_score / rubric_max_score) * 100
    if rubric_max_score > 0:
        score = (raw_score / rubric_max_score) * 100
    else:
        raise ValueError("rubric_max_score must be greater than 0")

    # Clamp the final score to 0-100 range
    score = max(0, min(100, score))

    # Return the score
    return score

In [ ]:
# Run the complete benchmark!
from tqdm import tqdm
import pandas as pd

print("🚀 Running benchmark...")
print("=" * 60)

# Step 1: Generate tutor bot responses for all test examples
print("\n📝 Step 1: Generating tutor bot responses...")
tutor_responses = []
for example in tqdm(test_examples, desc="Generating responses"):
    response = generate_tutor_response(example)
    tutor_responses.append(response)

# Step 2: Get judge evaluations for all responses
print("\n⚖️  Step 2: Evaluating responses with judge model...")
scores = []
for example, response in tqdm(zip(test_examples, tutor_responses), total=len(test_examples), desc="Evaluating"):
    score = evaluate_response(example, response)
    scores.append(score)

# Step 3: Calculate final score
print("\n📊 Step 3: Calculating results...")
average_score = sum(scores) / len(scores) if scores else 0

# Create summary table
results_data = []
for i, (example, response, score) in enumerate(zip(test_examples, tutor_responses, scores)):
    results_data.append({
        "Example #": i + 1,
        "Question": example.get("question", "N/A")[:50] + "..." if len(example.get("question", "")) > 50 else example.get("question", "N/A"),
        "Score": f"{score:.1f}/100"
    })

results_df = pd.DataFrame(results_data)
print("\n" + "=" * 60)
print("📈 BENCHMARK RESULTS")
print("=" * 60)
print(results_df.to_string(index=False))
print("\n" + "=" * 60)
print(f"🎯 FINAL SCORE: {average_score:.1f} / 100")
print("=" * 60)

## Summary

Building the benchmark made the alignment problem concrete. When I first stress-tested the tutor,
it was easy to extract answers — a hint would essentially give it away — so I iterated on the
prompt to tighten that path while keeping it encouraging. The rubric captured what I cared about
(especially *not* giving away answers and maintaining a supportive tone), and the LLM-judge let me
compare behavior across question types. The limits are real too: benchmarking can miss how a tutor
adapts to students of very different skill levels, which is exactly the kind of failure mode a
production evaluation suite would need to add.